## Cálculo dos indicadores

Nesse notebook são calculadas as variáveis de percentual de votos e de famílias no PBF para a análise

In [11]:
from utils.load_csv import load_csv
from utils.save_csv import save_csv

In [12]:
df_tse_filtrado = load_csv('resultados_2026_filtrado.csv')

df_observasampa_filtrado = load_csv('observasampa_filtrado.csv')

1 - Percentual de votos válidos dos candidatos a presidente no segundo turno por seção eleitoral do município de São Paulo

In [13]:
VOTO_BRANCO, VOTO_NULO = 95, 96

candidatos = df_tse_filtrado[~df_tse_filtrado["NR_VOTAVEL"].isin([VOTO_BRANCO, VOTO_NULO])]
nomes = list(candidatos["NM_VOTAVEL"].unique())

chaves = ["NR_ZONA", "NR_SECAO", "NR_LOCAL_VOTACAO", "NM_LOCAL_VOTACAO"]
df_votos_secao = (
    candidatos
    .pivot_table(index=chaves, columns="NM_VOTAVEL", values="QT_VOTOS",
                 aggfunc="sum", fill_value=0)
    .reset_index()
)
df_votos_secao.columns.name = None

df_votos_secao["TOTAL_VOTOS_VALIDOS"] = df_votos_secao[nomes].sum(axis=1)
total = df_votos_secao["TOTAL_VOTOS_VALIDOS"].where(df_votos_secao["TOTAL_VOTOS_VALIDOS"] > 0)
for nome in nomes:
    df_votos_secao[f"PCT_{nome}"] = (df_votos_secao[nome] / total * 100).round(2)

df_votos_secao = df_votos_secao.sort_values(["NR_ZONA", "NR_SECAO"]).reset_index(drop=True)

2 - Indicador de famílias beneficiárias do PBF por mil habitantes por distrito

In [14]:
df_observasampa_filtrado['pbf_mil_hab'] = (df_observasampa_filtrado['familias_pbf'] / df_observasampa_filtrado['pop_total']) * 1000

In [15]:
save_csv(df_votos_secao, 'votos_secao.csv')
save_csv(df_observasampa_filtrado, 'df_observasampa_filtrado')

Base salva em data\votos_secao.csv
Base salva em data\df_observasampa_filtrado
